In [ ]:
FIGURE = 'Fig.4'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.stats import gaussian_kde
settings = json.loads((INPUT_DIR / 'plot_settings.json').read_text(encoding='utf-8'))
shap_selected = pd.read_csv(INPUT_DIR / 'shap_values.csv').to_numpy()
X_selected = pd.read_csv(INPUT_DIR / 'feature_values.csv').to_numpy()
names = pd.read_csv(INPUT_DIR / 'shap_values.csv', nrows=0).columns.tolist()
ranking = pd.read_csv(INPUT_DIR / 'importance.csv').set_index('variable')
sel_importances = ranking.loc[names, 'importance'].to_numpy()
sel_labels = settings['labels']
colors = settings['colors']
pdp_results = [{'average':np.asarray(p['average']), 'grid_values':[np.asarray(a) for a in p['grid_values']]} for p in settings['pdp']]
n_top = len(names)
output_suffix = 'SPEI_MAM_Temperate'

def fix_co2_label(label):
    return label.replace('CO2', 'CO₂')
np.random.seed(42)
fixed_labels = [fix_co2_label(lb) for lb in sel_labels]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 8), gridspec_kw={'width_ratios': [1, 1], 'wspace': 0.05})
y_positions = np.arange(n_top)[::-1]
violin_width = 0.8
all_scatter_x = []
all_scatter_y = []
all_scatter_c = []
np.random.seed(42)

def get_violin_boundary(shap_vals, violin_width=0.8, resolution=100):
    from scipy import stats as sp_stats
    kde = sp_stats.gaussian_kde(shap_vals, bw_method=0.3)
    x_min, x_max = (shap_vals.min(), shap_vals.max())
    x_range = x_max - x_min if x_max > x_min else 1.0
    x_grid = np.linspace(x_min - x_range * 0.1, x_max + x_range * 0.1, resolution)
    density = kde(x_grid)
    max_density = density.max() if density.max() > 0 else 1.0
    normalized_density = density / max_density * (violin_width / 2)
    return (x_grid, normalized_density)
for i in range(n_top):
    y_pos = y_positions[i]
    shap_vals = shap_selected[:, i]
    feature_vals = X_selected[:, i]
    violin_parts = ax1.violinplot([shap_vals], positions=[y_pos], widths=[violin_width], vert=False, showmeans=False, showmedians=False, showextrema=False, bw_method=0.3)
    for pc in violin_parts['bodies']:
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.3)
        pc.set_edgecolor('#666666')
        pc.set_linewidth(1.0)
    normalized_vals = (feature_vals - feature_vals.min()) / (feature_vals.max() - feature_vals.min() + 1e-08)
    sample_indices = np.random.choice(len(shap_vals), min(120, len(shap_vals)), replace=False)
    shap_sample = shap_vals[sample_indices]
    norm_sample = normalized_vals[sample_indices]
    x_grid, density_half_width = get_violin_boundary(shap_vals, violin_width)
    y_jitter_list = []
    for shap_val in shap_sample:
        closest_idx = np.argmin(np.abs(x_grid - shap_val))
        max_jitter = density_half_width[closest_idx] * 0.85
        if max_jitter > 0.01:
            jitter = np.random.uniform(-max_jitter, max_jitter)
        else:
            jitter = np.random.uniform(-0.05, 0.05)
        y_jitter_list.append(jitter)
    y_jitter = np.array(y_jitter_list)
    all_scatter_x.extend(shap_sample)
    all_scatter_y.extend(y_pos + y_jitter)
    all_scatter_c.extend(norm_sample)
scatter = ax1.scatter(all_scatter_x, all_scatter_y, c=all_scatter_c, cmap='coolwarm', s=15, alpha=0.9, edgecolors='white', linewidth=0.3, zorder=5)
ax1.set_yticks(y_positions)
ax1.set_yticklabels([fixed_labels[i] for i in range(n_top)], fontsize=14)
ax1.set_xlabel('SHAP value', fontsize=14, labelpad=10)
ax1.axvline(x=0, color='black', linestyle='-', alpha=0.7, linewidth=1.5)
ax1.grid(axis='x', alpha=0.3, linestyle='--')
ax1.set_axisbelow(True)
for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_color('#555555')
limit = max(float(np.nanmax(np.abs(shap_selected))) * 1.05, 1e-6)
ax1.set_xlim(-limit, limit)
ax1.set_xticks([-limit, 0, limit])
ax1.set_xticklabels([f'{-limit:.3g}', '', f'{limit:.3g}'])
ax1.tick_params(axis='x', labelsize=12, pad=10)
ax1.tick_params(axis='y', labelsize=14, pad=14)
cax = inset_axes(ax1, width='3%', height='25%', loc='lower left', bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax1.transAxes)
cbar = plt.colorbar(scatter, cax=cax, orientation='vertical')
cbar.ax.yaxis.set_label_coords(3, 0.5)
cbar.set_label('Feature value', fontsize=11)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['Low', 'High'])
cbar.ax.tick_params(labelsize=11)
bar_height = 0.7
bars = ax2.barh(y_positions, sel_importances, height=bar_height, color=colors, alpha=0.7, edgecolor='white', linewidth=1)
max_importance = sel_importances.max() if len(sel_importances) > 0 else 0.2
ax2.set_xlim(0, 0.23)
ax2.set_xticks([0.05, 0.1, 0.15, 0.2])
ax2.set_xticklabels(['0.05', '0.10', '0.15', '0.20'])
ax2.tick_params(axis='x', labelsize=12, pad=10)
ax2.set_xlabel('Importance', fontsize=14, labelpad=10)
for i, (bar, pdp_result) in enumerate(zip(bars, pdp_results)):
    bar_width = bar.get_width()
    bar_y = bar.get_y()
    bar_height_actual = bar.get_height()
    try:
        pdp_values = pdp_result['average'][0]
        grid_values = pdp_result['grid_values'][0]
    except Exception:
        pdp_values = pdp_result.average[0]
        grid_values = pdp_result.grid_values[0]
    pdp_normalized = (pdp_values - pdp_values.min()) / (pdp_values.max() - pdp_values.min() + 1e-08)
    pdp_scaled = pdp_normalized * bar_height_actual * 0.6
    x_normalized = (grid_values - grid_values.min()) / (grid_values.max() - grid_values.min() + 1e-08)
    x_scaled = x_normalized * bar_width * 0.85
    y_offset = bar_y + bar_height_actual / 2 - pdp_scaled.mean()
    y_scaled = pdp_scaled + y_offset
    ax2.plot(x_scaled, y_scaled, color='black', linewidth=1.8, alpha=0.8)
ax2.set_yticks(y_positions)
ax2.set_yticklabels([])
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_color('#2c3e50')
for imp, bar in zip(sel_importances, bars):
    ax2.text(bar.get_width() + max_importance * 0.03, bar.get_y() + bar.get_height() / 2, f'{imp:.3f}', va='center', ha='left', fontsize=11)
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.set_axisbelow(True)
plt.tight_layout()
save_path_png = str(OUTPUT_DIR / 'Fig.4_600dpi.png')
plt.savefig(save_path_png, dpi=600, bbox_inches='tight', pad_inches=0.2)
print(f'💾 figure_saved_to: {save_path_png}')
plt.show()
